# MedGemma 4B - Google Colab Deployment

Serves MedGemma 4B as an HTTP API that your FastAPI backend calls.

**Before running:**
1. Runtime > Change runtime type > T4 GPU
2. Have ready:
   - Hugging Face token (with MedGemma access)
   - ngrok auth token (free from https://dashboard.ngrok.com/get-started/your-authtoken)

**After running all cells:** Copy the ngrok URL and set `MEDGEMMA_URL=<url>` in your backend `.env`

**Colab limitation:** Session dies after ~90 min idle or ~12 hrs max. Use Kaggle notebook for longer sessions.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate flask pyngrok huggingface_hub

In [ ]:
# Cell 2: Set your tokens
# Option A: Paste directly (quick but less secure)
# HF_TOKEN = "hf_..."
# NGROK_AUTH_TOKEN = "..."

# Option B: Use Colab secrets (recommended)
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")

from huggingface_hub import login
login(token=HF_TOKEN)
print("HF login OK")

In [ ]:
# Cell 3: Load MedGemma model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/medgemma-4b-it"  # instruction-tuned variant

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print(f"Model loaded on {model.device}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
# Cell 4: Prompt builder (mirrors backend/app/generation/llm.py)

def build_prompt(query: str, context: str, language: str, context_found: bool) -> str:
    lang_name = "Bahasa Melayu" if language == "ms" else "English"

    if context_found and context:
        return (
            f"You are a helpful medical assistant. "
            f"Answer using ONLY this context. Cite as [Source N: Title]. "
            f"Answer in {lang_name}.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {query}"
        )
    else:
        return (
            f"You are a helpful medical assistant. "
            f"Answer using medical knowledge. No citations needed. "
            f"Answer in {lang_name}.\n\n"
            f"Question: {query}"
        )

print("Prompt builder ready")

In [ ]:
# Cell 5: Generation function

def generate(query: str, context: str, language: str, context_found: bool) -> str:
    prompt = build_prompt(query, context, language, context_found)

    messages = [
        {"role": "user", "content": prompt}
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
        )

    # Decode only the new tokens (skip the input)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer

# Quick test
test_answer = generate("What is diabetes?", "", "en", False)
print("Test answer:", test_answer[:200], "...")

In [ ]:
# Cell 6: Flask server + ngrok tunnel
import threading
from flask import Flask, request, jsonify
from pyngrok import ngrok

app = Flask(__name__)

@app.route("/generate", methods=["POST"])
def generate_endpoint():
    data = request.get_json()
    try:
        answer = generate(
            query=data["query"],
            context=data.get("context", ""),
            language=data.get("language", "en"),
            context_found=data.get("context_found", False),
        )
        return jsonify({"answer": answer})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy", "model": MODEL_ID})

# Start ngrok tunnel
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000)

print("=" * 60)
print(f"MEDGEMMA_URL = {public_url}")
print("=" * 60)
print("Copy the URL above into your backend .env file.")
print("Colab sessions timeout after ~90 min idle.")

# Run Flask in a background thread so the cell doesn't block
threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=5000),
    daemon=True,
).start()

print("\nServer is running. Do NOT stop or re-run this cell.")

In [ ]:
# Cell 7: Test the endpoint locally
import requests

resp = requests.post(
    f"{public_url}/generate",
    json={
        "query": "Apakah ubat kencing manis?",
        "context": "",
        "language": "ms",
        "context_found": False,
    },
)
print(f"Status: {resp.status_code}")
print(f"Answer: {resp.json()['answer'][:300]}...")

In [ ]:
# Cell 8: Keep alive - run this to prevent idle timeout
import time

print("Keep-alive loop started. Press stop button to end.")
while True:
    try:
        r = requests.get(f"{public_url}/health")
        print(f"[{time.strftime('%H:%M:%S')}] Health: {r.status_code}")
    except Exception as e:
        print(f"[{time.strftime('%H:%M:%S')}] Ping failed: {e}")
    time.sleep(300)  # 5 minutes